# Soup development preflight

Synthetic GPU comparisons and the official Soup regression suite. This notebook does **not** run measured training or establish admission. Its parent uses only the standard library; child compiler caches and temporary probe sources are removed. Reports are written under `SOUP_RESULTS/preflight` before a failed check raises. The measured four-cell notebook is a separate artifact.


In [ ]:
# Generated solution installer. Writes files only; does not import solution or run training.
# Payload SHA256: c7d4d4584ce22c6e597fb634a20bcd2a933e798ef528635d0ba9bb729016d1df
# Manifest SHA256: a4bb0fb70b5e544bb8b3f02993ea509e5a79d736ea1c63d8583a213558cfe216
def _soup_install_solution_c7d4d4584ce22c6e(payload_base64, expected_payload_sha256, expected_manifest_sha256):
    """Self-contained runtime; its source is embedded in the inserted code cell."""
    import base64
    import hashlib
    import json
    import os
    from pathlib import Path, PurePosixPath
    import stat
    import uuid

    def canonical(value):
        return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode("utf-8")

    def digest(data):
        return hashlib.sha256(data).hexdigest()

    raw_repo = os.environ.get("SOUP_REPO")
    if not raw_repo or not raw_repo.strip():
        raise RuntimeError("SOUP_REPO must be set by the service template before this cell runs")
    repo = Path(raw_repo)
    if not repo.is_absolute() or ".." in repo.parts:
        raise RuntimeError("SOUP_REPO must be an absolute repository path without '..'")

    payload_bytes = base64.b64decode(payload_base64, validate=True)
    if digest(payload_bytes) != expected_payload_sha256:
        raise RuntimeError("Embedded solution payload SHA256 mismatch")
    payload = json.loads(payload_bytes)
    if not isinstance(payload, dict) or set(payload) != {"schema_version", "files"}:
        raise ValueError("Invalid solution payload schema")
    if type(payload["schema_version"]) is not int or payload["schema_version"] != 1:
        raise ValueError("Unsupported solution payload version")
    if not isinstance(payload["files"], list) or not payload["files"]:
        raise ValueError("Solution payload must contain source files")

    decoded, manifest, names = [], [], set()
    for item in payload["files"]:
        if not isinstance(item, dict) or set(item) != {"path", "size_bytes", "sha256", "data_base64"}:
            raise ValueError("Invalid solution file record")
        name = item["path"]
        if not isinstance(name, str) or not name or "\\" in name or "\x00" in name:
            raise ValueError("Invalid solution path")
        path = PurePosixPath(name)
        if (path.is_absolute() or str(path) != name or len(path.parts) < 2
                or path.parts[0] != "solution" or any(p in (".", "..") for p in path.parts)
                or any(":" in p for p in path.parts) or path.suffix != ".py"):
            raise ValueError(f"Unsafe solution path: {name!r}")
        if name in names:
            raise ValueError(f"Duplicate solution path: {name}")
        names.add(name)
        data = base64.b64decode(item["data_base64"], validate=True)
        if type(item["size_bytes"]) is not int or item["size_bytes"] != len(data):
            raise ValueError(f"Source size mismatch: {name}")
        if item["sha256"] != digest(data):
            raise ValueError(f"Source SHA256 mismatch: {name}")
        manifest.append({key: item[key] for key in ("path", "size_bytes", "sha256")})
        decoded.append((path.parts[1:], data, name))
    for name in names:
        if any(str(parent) in names for parent in PurePosixPath(name).parents):
            raise ValueError(f"Conflicting file and directory paths: {name}")
    if digest(canonical(manifest)) != expected_manifest_sha256:
        raise RuntimeError("Embedded solution manifest SHA256 mismatch")

    # Directory descriptors plus O_NOFOLLOW prevent symlink traversal even
    # between checking a path and opening it. The service runtime is Linux.
    if not hasattr(os, "O_NOFOLLOW") or not hasattr(os, "O_DIRECTORY"):
        raise RuntimeError("Safe solution installation requires Linux directory descriptors")
    directory_flags = os.O_RDONLY | os.O_DIRECTORY | os.O_NOFOLLOW
    file_flags = os.O_RDONLY | os.O_NOFOLLOW

    def open_repo():
        descriptor = os.open(repo.anchor, directory_flags)
        try:
            for name in repo.parts[1:]:
                next_descriptor = os.open(name, directory_flags, dir_fd=descriptor)
                os.close(descriptor)
                descriptor = next_descriptor
            return descriptor
        except BaseException:
            os.close(descriptor)
            raise

    def open_parent(solution_descriptor, parts, create):
        descriptor = os.dup(solution_descriptor)
        try:
            for name in parts[:-1]:
                if create:
                    try:
                        os.mkdir(name, mode=0o755, dir_fd=descriptor)
                    except FileExistsError:
                        pass
                try:
                    next_descriptor = os.open(name, directory_flags, dir_fd=descriptor)
                except FileNotFoundError:
                    if not create:
                        os.close(descriptor)
                        return None
                    raise
                os.close(descriptor)
                descriptor = next_descriptor
            return descriptor
        except BaseException:
            os.close(descriptor)
            raise

    def check_leaf(parent, leaf, name):
        try:
            mode = os.stat(leaf, dir_fd=parent, follow_symlinks=False).st_mode
        except FileNotFoundError:
            return
        if not stat.S_ISREG(mode):
            raise RuntimeError(f"Destination conflict (symlink or non-file): {name}")

    def matches(parent, leaf, data):
        try:
            descriptor = os.open(leaf, file_flags, dir_fd=parent)
        except FileNotFoundError:
            return False
        with os.fdopen(descriptor, "rb") as stream:
            status = os.fstat(stream.fileno())
            if not stat.S_ISREG(status.st_mode):
                raise RuntimeError(f"Destination is not a regular file: {leaf}")
            if status.st_size != len(data):
                return False
            actual = hashlib.sha256()
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                actual.update(chunk)
            return actual.hexdigest() == digest(data)

    repo_descriptor = open_repo()
    solution_descriptor = None
    written = unchanged = 0
    try:
        try:
            os.mkdir("solution", mode=0o755, dir_fd=repo_descriptor)
        except FileExistsError:
            pass
        solution_descriptor = os.open("solution", directory_flags, dir_fd=repo_descriptor)
        # Check every existing destination before replacing any file.
        for parts, data, name in decoded:
            parent = open_parent(solution_descriptor, parts, create=False)
            if parent is not None:
                try:
                    check_leaf(parent, parts[-1], name)
                finally:
                    os.close(parent)
        for parts, data, name in decoded:
            parent = open_parent(solution_descriptor, parts, create=True)
            temporary = None
            try:
                check_leaf(parent, parts[-1], name)
                if matches(parent, parts[-1], data):
                    unchanged += 1
                    continue
                temporary = ".soup-install-" + uuid.uuid4().hex + ".tmp"
                descriptor = os.open(temporary, os.O_WRONLY | os.O_CREAT | os.O_EXCL | os.O_NOFOLLOW,
                                     0o644, dir_fd=parent)
                with os.fdopen(descriptor, "wb") as stream:
                    stream.write(data)
                    stream.flush()
                    os.fsync(stream.fileno())
                check_leaf(parent, parts[-1], name)
                os.replace(temporary, parts[-1], src_dir_fd=parent, dst_dir_fd=parent)
                temporary = None
                written += 1
            finally:
                if temporary is not None:
                    try:
                        os.unlink(temporary, dir_fd=parent)
                    except FileNotFoundError:
                        pass
                os.close(parent)
        # Verify every installed file against the hashed manifest, including
        # files that were already identical and therefore not rewritten.
        for parts, data, name in decoded:
            parent = open_parent(solution_descriptor, parts, create=False)
            if parent is None:
                raise RuntimeError(f"Installed source directory disappeared: {name}")
            try:
                check_leaf(parent, parts[-1], name)
                if not matches(parent, parts[-1], data):
                    raise RuntimeError(f"Installed source SHA256 mismatch: {name}")
            finally:
                os.close(parent)
    finally:
        if solution_descriptor is not None:
            os.close(solution_descriptor)
        os.close(repo_descriptor)
    result = {"payload_sha256": expected_payload_sha256,
              "manifest_sha256": expected_manifest_sha256,
              "files": len(decoded), "written": written, "unchanged": unchanged}
    print(json.dumps({"soup_solution_install": result}, sort_keys=True))
    return result

_soup_install_solution_c7d4d4584ce22c6e(
    '...',
)
del _soup_install_solution_c7d4d4584ce22c6e


In [ ]:
def _run_preflight(solution_manifest, encoded_dev_files):
    """Standalone notebook runtime: no Torch imports in the notebook parent."""
    import base64
    import hashlib
    import json
    import os
    from pathlib import Path
    import signal
    import subprocess
    import sys
    import tempfile
    import time
    import traceback

    repo = Path(os.environ["SOUP_REPO"])
    reports = Path(os.environ["SOUP_RESULTS"]) / "preflight"
    reports.mkdir(parents=True, exist_ok=True)

    def write_json(path, value):
        path.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")

    def retire(process, include_descendants):
        # The GPU runner creates a separate process group for each probe.
        # On an outer timeout, collect and stop those descendants as well.
        groups = {process.pid}
        discovery_error = None
        try:
            if include_descendants:
                listing = subprocess.run(
                    ["ps", "-eo", "pid=,ppid=,pgid="], capture_output=True,
                    text=True, timeout=5, check=True,
                )
                rows = [tuple(map(int, line.split())) for line in listing.stdout.splitlines()]
                descendants = {process.pid}
                while True:
                    found = {pid for pid, parent, _ in rows if parent in descendants}
                    if found <= descendants:
                        break
                    descendants.update(found)
                groups.update(group for pid, _, group in rows if pid in descendants)
        except Exception as error:
            # Still stop the known group if process discovery itself fails.
            discovery_error = f"{type(error).__name__}: {error}"
        groups.discard(os.getpgrp())
        for group in groups:
            try:
                os.killpg(group, signal.SIGKILL)
            except ProcessLookupError:
                pass
        if process.poll() is None:
            process.kill()
        return groups, discovery_error

    def child(name, command, cwd, environment, timeout):
        started = time.monotonic()
        record = {"name": name, "command": command, "timeout_seconds": timeout}
        stdout = stderr = ""
        process = None
        groups = set()
        interrupted = None
        try:
            process = subprocess.Popen(
                command, cwd=cwd, env=environment, start_new_session=True,
                stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                text=True, encoding="utf-8", errors="replace",
            )
            try:
                stdout, stderr = process.communicate(timeout=timeout)
                record["state"] = "passed" if process.returncode == 0 else "failed"
            except subprocess.TimeoutExpired:
                record["state"] = "timeout"
                groups, cleanup_error = retire(process, include_descendants=True)
                if cleanup_error:
                    record["cleanup_error"] = cleanup_error
                stdout, stderr = process.communicate(timeout=15)
            finally:
                # Normal completion should leave no descendants. Retire the
                # original group too, in case a child retained that group.
                if process.poll() is not None:
                    retired, _ = retire(process, include_descendants=False)
                    groups.update(retired)
            record["returncode"] = process.returncode
        except BaseException as error:
            record.update(state="error", error=f"{type(error).__name__}: {error}")
            if process is not None:
                try:
                    retired, cleanup_error = retire(process, include_descendants=True)
                    groups.update(retired)
                    if cleanup_error:
                        record["cleanup_error"] = cleanup_error
                    stdout, stderr = process.communicate(timeout=15)
                    record["returncode"] = process.returncode
                except BaseException as cleanup_error:
                    record["cleanup_error"] = f"{type(cleanup_error).__name__}: {cleanup_error}"
            if not isinstance(error, Exception):
                record["interrupted"] = True
                interrupted = error
        record["elapsed_seconds"] = time.monotonic() - started
        record["process_reaped"] = process is None or process.poll() is not None
        record["retired_process_groups"] = sorted(groups)
        (reports / f"{name}.stdout.log").write_text(stdout, encoding="utf-8")
        (reports / f"{name}.stderr.log").write_text(stderr, encoding="utf-8")
        write_json(reports / f"{name}.json", record)
        if interrupted is not None:
            raise interrupted
        if not record["process_reaped"] or "cleanup_error" in record:
            raise RuntimeError(f"Could not confirm cleanup for {name}; inspect its saved status")
        return record

    summary = {"purpose": "development preflight only; no measured training or admission result",
               "notebook_parent_imports_torch": False, "state": "failed"}
    workspace = None
    try:
        # Needed only for exceptional cleanup of the runner's nested groups.
        # Check availability before starting any regression or GPU subprocess.
        subprocess.run(["ps", "-eo", "pid=,ppid=,pgid="], capture_output=True,
                       text=True, timeout=5, check=True)
        with tempfile.TemporaryDirectory(prefix="duo-preflight-", dir="/kaggle/working") as temporary:
            workspace = Path(temporary)
            for item in solution_manifest:
                data = (repo / item["path"]).read_bytes()
                if len(data) != item["size_bytes"] or hashlib.sha256(data).hexdigest() != item["sha256"]:
                    raise RuntimeError(f"Installed solution differs from preflight manifest: {item['path']}")
                destination = workspace / item["path"]
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(data)
            dev_files = json.loads(base64.b64decode(encoded_dev_files, validate=True))
            for item in dev_files:
                data = base64.b64decode(item["data_base64"], validate=True)
                if hashlib.sha256(data).hexdigest() != item["sha256"] or len(data) != item["size_bytes"]:
                    raise RuntimeError("Embedded development source hash mismatch")
                name = item["name"]
                if Path(name).name != name or not name.endswith(".py"):
                    raise RuntimeError("Unsafe development source name")
                destination = workspace / "dev" / name
                destination.parent.mkdir(exist_ok=True)
                destination.write_bytes(data)

            regression_env = dict(os.environ)
            regression_env.update(
                SOUP_APPLY_SOLUTION="1",
                TRITON_CACHE_DIR=str(workspace / "regression-cache" / "triton"),
                TORCHINDUCTOR_CACHE_DIR=str(workspace / "regression-cache" / "inductor"),
                PYTHONUNBUFFERED="1",
            )
            summary["regression"] = child(
                "regression", [sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                repo, regression_env, 600,
            )
            probe_env = dict(os.environ, PYTHONUNBUFFERED="1")
            # The existing runner assigns fresh caches to each probe child.
            summary["gpu_runner"] = child(
                "gpu_runner", [sys.executable, str(workspace / "dev" / "run_gpu_probes.py"),
                               "--out", str(reports / "gpu"), "--timeout", "300"],
                workspace, probe_env, 1050,
            )
            if all(summary[name]["state"] == "passed" and summary[name]["process_reaped"]
                   for name in ("regression", "gpu_runner")):
                summary["state"] = "passed"
    except BaseException as error:
        summary["error"] = f"{type(error).__name__}: {error}"
        (reports / "preflight_error.log").write_text(traceback.format_exc(), encoding="utf-8")
    finally:
        summary["temporary_workspace_removed"] = workspace is None or not workspace.exists()
        if not summary["temporary_workspace_removed"]:
            summary["state"] = "failed"
        write_json(reports / "summary.json", summary)
    print(json.dumps(summary, indent=2))
    if summary["state"] != "passed":
        raise RuntimeError("Development preflight failed; inspect SOUP_RESULTS/preflight reports")

_run_preflight(
    [{'path': 'solution/__init__.py', 'size_bytes': 157, 'sha256': '59ddcd5624b285618fbd45fca494c717b6f3dd9bab329ef78d8faaca2a56abea'}, {'path': 'solution/metrics.py', 'size_bytes': 12135, 'sha256': '50352c62c4eb0e2f6135a90c04ac435bcfc7ecf95ab8b2c3e059a0ef4db21998'}, {'path': 'solution/mlp_storage.py', 'size_bytes': 6673, 'sha256': '709c2b293c4d4dfc9abedff2124fac8b6730c752708eb77436564bd3c9889119'}, {'path': 'solution/my_kernels.py', 'size_bytes': 546, 'sha256': 'fcebd11a9167689207e79f83e72cc75f58869ccde1df1a7c828ff00fc7bb96cc'}, {'path': 'solution/rmsnorm.py', 'size_bytes': 5718, 'sha256': '4bd55d0f3508a3f4b6151c97ae25e9c042898c094cbdd5ccfc46aa92f7af9488'}, {'path': 'solution/swiglu.py', 'size_bytes': 3185, 'sha256': '...',
)
